# 06 · Ca cảnh báo Khánh đề xuất: 22/11/2018

Đối chiếu ca FN 08:15 và TP 09:00 của bản HGB tổng giờ **11/10** với bản phân vị **14/10**. Không gọi các giá trị trong cột actual là số đo đã biết ở origin: chúng là tổng 4 số đo tương lai dùng đánh giá offline.

In [1]:
from pathlib import Path
import sys
import json
ROOT = Path.cwd()
if not (ROOT / "src/steel").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src/steel").is_dir():
    raise FileNotFoundError("Mở notebook từ repo hoặc notebooks/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from src.steel.preprocessing import (load_official_raw, build_supervised_rows,
    split_supervised_rows, BASE_FEATURES, TARGET)
from src.steel.duy_forecasting import forecast_features
raw, audit = load_official_raw(ROOT / "data/steel/raw/Steel_industry_data.csv")
OLD = ROOT / "reports/steel/release_2026-10-11"
NEW = ROOT / "reports/steel/release_2026-10-14"
old = pd.read_csv(OLD / "test_predictions.csv", parse_dates=["observation_time"])
new = pd.read_csv(NEW / "test_predictions.csv", parse_dates=["issue_time"])
policy = json.loads((OLD / "locked_policy.json").read_text(encoding="utf-8"))
manifest = json.loads((NEW / "manifest.json").read_text(encoding="utf-8"))
threshold = policy["threshold_kWh"]

## 1. Lấy đúng các dòng CSV và phân loại

Cùng ngưỡng TRAIN, giữ riêng biên buffer cũ và dải quantile mới. Đây là ca hồi cứu chọn để giải thích; không dùng nó để sửa model/margin.

In [2]:
view = old.loc[old.observation_time.between("2018-11-22 07:30", "2018-11-22 09:30")].copy()
view = view.merge(new, left_on="observation_time", right_on="issue_time", validate="one_to_one")
view["legacy_alert"] = view.upper_hgb_31_leaves_kWh > threshold
view["new_alert"] = view.hour_total_calibrated_upper_kWh > threshold
def outcome(actual, alert):
    return "TP" if actual > threshold and alert else "FN" if actual > threshold else "FP" if alert else "TN"
view["legacy_case"] = [outcome(y, a) for y, a in zip(view.target_next_60m_kWh, view.legacy_alert)]
view["new_case"] = [outcome(y, a) for y, a in zip(view.hour_total_actual_kWh, view.new_alert)]
display(view[["observation_time", "target_next_60m_kWh", "pred_hgb_31_leaves_kWh", "upper_hgb_31_leaves_kWh", "legacy_case", "hour_total_p50_kWh", "hour_total_calibrated_upper_kWh", "new_case"]])
assert view.loc[view.observation_time.eq("2018-11-22 08:15"), "legacy_case"].item() == "FN"
assert view.loc[view.observation_time.eq("2018-11-22 09:00"), "legacy_case"].item() == "TP"

,observation_time,target_next_60m_kWh,pred_hgb_31_leaves_kWh,upper_hgb_31_leaves_kWh,legacy_case,hour_total_p50_kWh,hour_total_calibrated_upper_kWh,new_case
0,2018-11-22 07:30:00,154.08,136.462697,173.014378,TN,108.282417,197.924231,TN
1,2018-11-22 07:45:00,269.96,270.674471,307.226152,TN,223.734160,307.277836,TN
2,2018-11-22 08:00:00,388.15,343.395102,379.946783,TP,241.344771,397.142581,TP
3,2018-11-22 08:15:00,486.25,321.218167,357.769848,FN,283.590754,413.411933,TP
4,2018-11-22 08:30:00,505.59,385.772956,422.324637,TP,364.129757,446.365426,TP
5,2018-11-22 08:45:00,542.10,403.209827,439.761509,TP,391.200491,459.468563,TP
6,2018-11-22 09:00:00,554.01,383.497585,420.049266,TP,397.046529,444.772314,TP
7,2018-11-22 09:15:00,553.97,390.739434,427.291115,TP,414.885955,463.038914,TP
8,2018-11-22 09:30:00,536.65,394.897936,431.449617,TP,398.344152,449.110195,TP


In [3]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(view.observation_time, view.target_next_60m_kWh, marker="o", label="Measured future hour total (offline)")
ax.plot(view.observation_time, view.upper_hgb_31_leaves_kWh, label="Legacy HGB + buffer")
ax.plot(view.observation_time, view.hour_total_calibrated_upper_kWh, label="New quantile calibrated upper")
ax.axhline(threshold, color="red", ls="--", label="TRAIN p95; research threshold")
ax.set(ylabel="Next-hour energy (kWh)", title="22 Nov 2018: retrospective alert cases")
ax.legend(); fig.autofmt_xdate(); fig.tight_layout(); plt.show()

C:\Users\dzyuu\AppData\Local\Temp\ipykernel_20672\1081067005.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(); fig.autofmt_xdate(); fig.tight_layout(); plt.show()


## 2. Quyết định sau cảnh báo

Người xem kiểm tra đồng hồ/lịch vận hành thực nếu có, ghi nhận lý do và kết quả. Demo chỉ lưu thao tác kiểm tra, không điều khiển máy. Model có thể bỏ sót tải tăng nhanh; chưa biết nguyên nhân vật lý từ dataset. Thay model có thể đổi FP/FN; không bảo đảm loại bỏ mọi bỏ sót, không chứng minh tiết kiệm hoặc lead time thực.